# 💳 Credit Card Fraud Detection — ML Models
### Random Forest · Logistic Regression · XGBoost

---

This notebook trains and evaluates **three ML models** on the Credit Card Fraud Detection dataset (Kaggle).  
Dataset: **284,807 transactions** | **492 frauds** (~0.17%) — highly imbalanced.

**Pipeline:**
1. Install dependencies
2. Load `creditcard.csv`
3. EDA
4. Preprocessing (RobustScaler, stratified 70/15/15 split, NO data leakage)
5. Train Logistic Regression
6. Train Random Forest
7. Train XGBoost
8. Compare metrics
9. Visualizations
10. Save models

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib kaggle --quiet
print('✅ All packages installed successfully!')

## 📥 Step 2 — Load Dataset

**Option A:** Upload `creditcard.csv` manually (recommended for beginners)  
**Option B:** Auto-download from Kaggle (needs `kaggle.json` API token)

In [ ]:
# ── OPTION A: Upload creditcard.csv manually ─────────────────────────────
# Uncomment the three lines below, run this cell, then choose your file.
# from google.colab import files
# uploaded = files.upload()   # select creditcard.csv from your PC
# print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── OPTION B: Download from Kaggle automatically ─────────────────────────
# Upload your kaggle.json first, then run this cell.
import os

if os.path.exists('kaggle.json'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.system('cp kaggle.json /root/.kaggle/kaggle.json')
    os.system('chmod 600 /root/.kaggle/kaggle.json')
    ret = os.system('kaggle datasets download -d mlg-ulb/creditcardfraud --unzip -q')
    print('✅ Dataset downloaded!' if ret == 0 else '❌ Kaggle download failed.')
elif os.path.exists('/root/.kaggle/kaggle.json'):
    os.system('kaggle datasets download -d mlg-ulb/creditcardfraud --unzip -q')
    print('✅ Dataset downloaded from Kaggle!')
elif os.path.exists('creditcard.csv'):
    print('✅ creditcard.csv already present.')
else:
    print('⚠️  creditcard.csv not found.')
    print('   → Use OPTION A above to upload the CSV manually, OR')
    print('   → Upload your kaggle.json and re-run this cell.')

## 📊 Step 3 — Imports & EDA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler        # ← matches project pipeline.py
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
import xgboost as xgb

# ── Dark theme for all plots ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444466',
    'axes.labelcolor':  '#ccccee',
    'xtick.color':      '#aaaacc',
    'ytick.color':      '#aaaacc',
    'text.color':       '#e0e0ff',
    'grid.color':       '#2a2a4a',
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#444466',
    'font.family':      'DejaVu Sans',
})
PALETTE = ['#6c63ff', '#ff6584', '#43e97b']   # violet, pink, green
print('✅ Libraries imported. Versions:')
print(f'   scikit-learn : {__import__("sklearn").__version__}')
print(f'   xgboost      : {xgb.__version__}')
print(f'   numpy        : {np.__version__}')
print(f'   pandas       : {pd.__version__}')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────
df = pd.read_csv('creditcard.csv')
print(f'Shape       : {df.shape}')
print(f'Columns     : {list(df.columns)}')
print(f'Dtypes OK   : {all(df.dtypes != object)}')
df.head()

In [ ]:
# ── Basic statistics ──────────────────────────────────────────────────────
total      = len(df)
fraud      = int(df['Class'].sum())
legit      = total - fraud
fraud_rate = fraud / total * 100

print('=' * 52)
print(f'  Total transactions : {total:>10,}')
print(f'  Fraudulent (1)     : {fraud:>10,}  ({fraud_rate:.4f}%)')
print(f'  Legitimate (0)     : {legit:>10,}')
print('=' * 52)
print(f'\n  Missing values     : {df.isnull().sum().sum()}')
print(f'  Duplicate rows     : {df.duplicated().sum()}')

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f0f1a')
counts = df['Class'].value_counts().sort_index()
labels = ['Legitimate (0)', 'Fraud (1)']
colors = ['#6c63ff', '#ff6584']

# Bar
axes[0].bar(labels, counts.values, color=colors, width=0.5,
            edgecolor='white', linewidth=0.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold',
                   color='#e0e0ff', pad=12)
axes[0].set_ylabel('Count', color='#ccccee')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 800, f'{v:,}', ha='center', va='bottom',
                 color='white', fontweight='bold')

# Donut
wedge_props = dict(width=0.55, edgecolor='#0f0f1a', linewidth=2)
axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.3f%%', startangle=140, pctdistance=0.75,
            wedgeprops=wedge_props,
            textprops={'color': '#e0e0ff', 'fontsize': 11})
axes[1].set_title('Class Ratio', fontsize=14, fontweight='bold',
                   color='#e0e0ff', pad=12)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()
print('⚠️  Highly imbalanced — accuracy alone is misleading!')

In [ ]:
# ── Amount distribution by class ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

for ax, cls, color, label in zip(
        axes, [0, 1], ['#6c63ff', '#ff6584'],
        ['Legitimate Transactions', 'Fraudulent Transactions']):
    subset = df[df['Class'] == cls]['Amount']
    ax.hist(subset, bins=60, color=color, alpha=0.85, edgecolor='none')
    ax.set_title(label, fontsize=13, fontweight='bold', color='#e0e0ff')
    ax.set_xlabel('Amount (USD)', color='#ccccee')
    ax.set_ylabel('Frequency', color='#ccccee')
    med = subset.median()
    ax.axvline(med, color='white', linestyle='--', linewidth=1.2,
               label=f'Median: ${med:.2f}')
    ax.legend(fontsize=9)

plt.suptitle('Transaction Amount Distribution', fontsize=15,
             fontweight='bold', color='#e0e0ff', y=1.01)
plt.tight_layout()
plt.savefig('amount_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()

## ⚙️ Step 4 — Preprocessing (No Data Leakage)

> **Critical:** Scaler is fitted **only on train set** and then applied to val/test.  
> This matches the project's `pipeline.py` and prevents data leakage.

In [ ]:
# ── Remove duplicates (matches project pipeline) ──────────────────────────
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate rows. Remaining: {len(df):,}')

# ── Feature/label split ───────────────────────────────────────────────────
FEATURES = [c for c in df.columns if c != 'Class']
X_raw = df[FEATURES].copy()
y     = df['Class'].values

print(f'Features : {len(FEATURES)}  → {FEATURES}')
print(f'Labels   : {y.shape}  |  Fraud: {y.sum()}  |  Legit: {(y==0).sum()}')

In [ ]:
# ── Stratified split: 70% train | 15% val | 15% test ─────────────────────
# Step 1: split off 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw.values, y, test_size=0.15, stratify=y, random_state=42
)
# Step 2: split remaining 85% into 70% train / 15% val
# 15/85 ≈ 0.1765
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42
)

print('Split summary (stratified)')
print('-' * 48)
for split_name, yy in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    n, f = len(yy), yy.sum()
    print(f'  {split_name:5s}: {n:>7,} samples | {f:>3} fraud ({f/n*100:.3f}%)')

In [ ]:
# ── Scale Time & Amount with RobustScaler ─────────────────────────────────
# V1-V28 are already PCA-transformed (zero mean, unit variance)
# Only 'Time' (col 0) and 'Amount' (col -1, index 29) need scaling

TIME_IDX   = FEATURES.index('Time')
AMOUNT_IDX = FEATURES.index('Amount')
SCALE_COLS = [TIME_IDX, AMOUNT_IDX]

scaler = RobustScaler()   # robust to outliers — matches project pipeline.py

# Fit ONLY on train, transform all
X_train[:, SCALE_COLS] = scaler.fit_transform(X_train[:, SCALE_COLS])
X_val  [:, SCALE_COLS] = scaler.transform(X_val  [:, SCALE_COLS])
X_test [:, SCALE_COLS] = scaler.transform(X_test [:, SCALE_COLS])

print(f'✅ RobustScaler fitted on train, applied to val & test.')
print(f'   Feature matrix shape: {X_train.shape}')
print(f'   Time  col idx : {TIME_IDX}')
print(f'   Amount col idx: {AMOUNT_IDX}')

## 📐 Helper Functions

In [ ]:
def evaluate_model(model, X, y, model_name='Model', threshold=0.5):
    """Full metric evaluation for a binary classifier."""
    # Get probabilities
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X)[:, 1]
    else:
        # Fallback: decision_function scaled to [0,1]
        scores = model.decision_function(X)
        y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        'accuracy' : accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall'   : recall_score(y, y_pred, zero_division=0),
        'f1'       : f1_score(y, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y, y_prob),
        'pr_auc'   : average_precision_score(y, y_prob),
    }

    print(f'\n{"="*58}')
    print(f'  {model_name} — Metrics  (threshold={threshold})')
    print(f'{"─"*58}')
    for k, v in metrics.items():
        filled = '█' * int(v * 30)
        empty  = '░' * (30 - int(v * 30))
        print(f'  {k:10s}: {v:.4f}  {filled}{empty}')
    print(f'{"="*58}')
    print(classification_report(y, y_pred, target_names=['Legit', 'Fraud'],
                                 digits=4))
    # Confusion matrix counts
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
    return metrics, y_prob, y_pred


def plot_confusion_matrix(y_true, y_pred, model_name, ax):
    """Plot a styled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='RdPu', ax=ax,
        linewidths=1.5, linecolor='#0f0f1a',
        annot_kws={'size': 15, 'weight': 'bold'},
        cbar=False
    )
    ax.set_title(model_name, fontsize=13, fontweight='bold',
                 color='#e0e0ff', pad=10)
    ax.set_xlabel('Predicted Label', color='#ccccee', fontsize=11)
    ax.set_ylabel('True Label', color='#ccccee', fontsize=11)
    # ← Fixed: use set_ticklabels on tick positions
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Legit', 'Fraud'], color='#ccccee')
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Legit', 'Fraud'], color='#ccccee', rotation=0)


print('✅ Helper functions defined.')

## 🔵 Step 5 — Logistic Regression (Baseline)

In [ ]:
print('Training Logistic Regression ...')

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # handles class imbalance
    C=0.01,                    # L2 regularization
    solver='lbfgs',
    random_state=42
    # Note: n_jobs is NOT valid for solver='lbfgs' — removed to prevent error
)
lr_model.fit(X_train, y_train)

print('✅ Logistic Regression trained!')
lr_metrics, lr_prob, lr_pred = evaluate_model(
    lr_model, X_test, y_test, 'Logistic Regression'
)

## 🟢 Step 6 — Random Forest

In [ ]:
print('Training Random Forest (may take ~1-2 min on Colab) ...')

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',   # handles class imbalance
    random_state=42,
    n_jobs=-1                  # use all CPU cores
)
rf_model.fit(X_train, y_train)

print('✅ Random Forest trained!')
rf_metrics, rf_prob, rf_pred = evaluate_model(
    rf_model, X_test, y_test, 'Random Forest'
)

## 🟡 Step 7 — XGBoost

In [ ]:
print('Training XGBoost ...')

# Compute class imbalance ratio for scale_pos_weight
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pw   = neg_count / pos_count
print(f'  Neg={neg_count:,}  Pos={pos_count}  scale_pos_weight={scale_pw:.1f}')

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pw,  # handles class imbalance
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='aucpr',        # PR-AUC as early-stop metric
    early_stopping_rounds=20,
    random_state=42,
    tree_method='hist',         # faster histogram method
    n_jobs=-1
    # Note: use_label_encoder removed — deprecated/removed in XGBoost >= 1.6
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

print(f'\n✅ XGBoost trained! Best iteration: {xgb_model.best_iteration}')
xgb_metrics, xgb_prob, xgb_pred = evaluate_model(
    xgb_model, X_test, y_test, 'XGBoost'
)

## 📈 Step 8 — Model Comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model'    : ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy' : [lr_metrics['accuracy'],  rf_metrics['accuracy'],  xgb_metrics['accuracy']],
    'Precision': [lr_metrics['precision'], rf_metrics['precision'], xgb_metrics['precision']],
    'Recall'   : [lr_metrics['recall'],    rf_metrics['recall'],    xgb_metrics['recall']],
    'F1'       : [lr_metrics['f1'],        rf_metrics['f1'],        xgb_metrics['f1']],
    'ROC-AUC'  : [lr_metrics['roc_auc'],   rf_metrics['roc_auc'],   xgb_metrics['roc_auc']],
    'PR-AUC'   : [lr_metrics['pr_auc'],    rf_metrics['pr_auc'],    xgb_metrics['pr_auc']],
})
results.set_index('Model', inplace=True)

# Highlight best in each column
styled = (
    results.style
    .background_gradient(cmap='RdYlGn', axis=0)
    .format('{:.4f}')
    .set_caption('📊 Model Comparison — Test Set')
    .set_properties(**{'text-align': 'center', 'font-size': '13px'})
)
display(styled)

In [ ]:
# ── Grouped bar chart ─────────────────────────────────────────────────────
metrics_cols = ['Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
x     = np.arange(len(metrics_cols))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0f0f1a')

for i, (mdl, color) in enumerate(zip(results.index, PALETTE)):
    vals = [results.loc[mdl, m] for m in metrics_cols]
    bars = ax.bar(x + i * width, vals, width, label=mdl,
                  color=color, alpha=0.85, edgecolor='white', linewidth=0.4)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.012,
                f'{h:.3f}', ha='center', va='bottom',
                fontsize=8, color='white', fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_cols, fontsize=12)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score', color='#ccccee', fontsize=12)
ax.set_title('Model Comparison — Key Metrics (Test Set)',
             fontsize=15, fontweight='bold', color='#e0e0ff', pad=15)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()

## 🔲 Step 9 — Visualizations
### 9a — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.patch.set_facecolor('#0f0f1a')

for ax, y_pred_arr, name in zip(
        axes,
        [lr_pred, rf_pred, xgb_pred],
        ['Logistic Regression', 'Random Forest', 'XGBoost']):
    plot_confusion_matrix(y_test, y_pred_arr, name, ax)

plt.suptitle('Confusion Matrices — Test Set', fontsize=16,
             fontweight='bold', color='#e0e0ff', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()

### 9b — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0f0f1a')

for prob, name, color in zip(
        [lr_prob, rf_prob, xgb_prob],
        ['Logistic Regression', 'Random Forest', 'XGBoost'],
        PALETTE):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, lw=2.5, color=color,
            label=f'{name}  (AUC = {auc_val:.4f})')
    ax.fill_between(fpr, tpr, alpha=0.05, color=color)

ax.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.4, label='Random (AUC = 0.5)')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Test Set', fontsize=15, fontweight='bold',
             color='#e0e0ff', pad=12)
ax.legend(fontsize=11, loc='lower right')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()

### 9c — Precision-Recall Curves
> **PR-AUC is the primary metric for imbalanced fraud detection.**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0f0f1a')

for prob, name, color in zip(
        [lr_prob, rf_prob, xgb_prob],
        ['Logistic Regression', 'Random Forest', 'XGBoost'],
        PALETTE):
    prec, rec, _ = precision_recall_curve(y_test, prob)
    pr_auc_val = average_precision_score(y_test, prob)
    ax.plot(rec, prec, lw=2.5, color=color,
            label=f'{name}  (PR-AUC = {pr_auc_val:.4f})')

# Baseline: a random classifier has precision = fraud prevalence
baseline = float(y_test.mean())
ax.axhline(baseline, color='white', linestyle='--', lw=1, alpha=0.5,
           label=f'Random Classifier ({baseline:.4f})')

ax.set_xlim([0, 1.01])
ax.set_ylim([0, 1.05])
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Test Set', fontsize=15,
             fontweight='bold', color='#e0e0ff', pad=12)
ax.legend(fontsize=11, loc='upper right')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()

### 9d — Feature Importance (Random Forest & XGBoost)

In [ ]:
top_n = 15
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0f0f1a')

for ax, model, name, color in zip(
        axes,
        [rf_model, xgb_model],
        ['Random Forest', 'XGBoost'],
        ['#6c63ff', '#43e97b']):
    imps = model.feature_importances_
    idx  = np.argsort(imps)[::-1][:top_n]
    top_feats = [FEATURES[i] for i in idx]
    top_imps  = imps[idx]

    ax.barh(top_feats[::-1], top_imps[::-1], color=color,
            alpha=0.85, edgecolor='none')
    for i, val in enumerate(top_imps[::-1]):
        ax.text(val + 0.001, i, f'{val:.4f}', va='center',
                color='white', fontsize=8)
    ax.set_xlabel('Importance Score', color='#ccccee')
    ax.set_title(f'{name} — Top {top_n} Features',
                 fontsize=13, fontweight='bold', color='#e0e0ff', pad=10)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importances', fontsize=16, fontweight='bold',
             color='#e0e0ff', y=1.01)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()

### 9e — Predicted Probability Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f1a')

for ax, prob, name in zip(
        axes,
        [lr_prob, rf_prob, xgb_prob],
        ['Logistic Regression', 'Random Forest', 'XGBoost']):
    legit_probs = prob[y_test == 0]
    fraud_probs = prob[y_test == 1]

    ax.hist(legit_probs, bins=60, alpha=0.7, color='#6c63ff',
            label=f'Legit  (n={len(legit_probs):,})', density=True)
    ax.hist(fraud_probs, bins=30, alpha=0.8, color='#ff6584',
            label=f'Fraud  (n={len(fraud_probs)})', density=True)
    ax.axvline(0.5, color='white', linestyle='--', lw=1.5,
               label='Decision threshold')
    ax.set_xlabel('P(Fraud)', color='#ccccee')
    ax.set_ylabel('Density', color='#ccccee')
    ax.set_title(name, fontsize=13, fontweight='bold', color='#e0e0ff')
    ax.legend(fontsize=9)

plt.suptitle('Predicted Probability Distributions', fontsize=15,
             fontweight='bold', color='#e0e0ff', y=1.02)
plt.tight_layout()
plt.savefig('probability_dist.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f1a')
plt.show()

## 💾 Step 10 — Save Models

In [ ]:
os.makedirs('saved_models', exist_ok=True)

joblib.dump(lr_model,  'saved_models/logistic_regression.pkl')
joblib.dump(rf_model,  'saved_models/random_forest.pkl')
joblib.dump(xgb_model, 'saved_models/xgboost.pkl')
joblib.dump(scaler,    'saved_models/robust_scaler.pkl')

# Verify files saved correctly
for fname in ['logistic_regression.pkl', 'random_forest.pkl',
              'xgboost.pkl', 'robust_scaler.pkl']:
    path = f'saved_models/{fname}'
    size = os.path.getsize(path) / 1024
    print(f'  ✅ {fname:<30}  {size:>8.1f} KB')

## 🚀 Step 11 — Inference Demo (Known Fraud Transaction)

In [ ]:
# Pick first fraud sample from test set
fraud_indices = np.where(y_test == 1)[0]
print(f'Total fraud cases in test set: {len(fraud_indices)}')

sample = X_test[fraud_indices[0]].reshape(1, -1)

print('\n' + '=' * 60)
print('  INFERENCE DEMO — Predicting a Known Fraud Transaction')
print('-' * 60)
for model, name in [
        (lr_model,  'Logistic Regression'),
        (rf_model,  'Random Forest'),
        (xgb_model, 'XGBoost')]:
    p = model.predict_proba(sample)[0, 1]
    verdict = '🚨 FRAUD DETECTED' if p >= 0.5 else '✅ Classified as Legit'
    print(f'  {name:22s}  P(fraud) = {p:.6f}   {verdict}')
print('=' * 60)

## 📋 Final Summary

| Model | Handles Imbalance | Speed | PR-AUC | Best for |
|---|---|---|---|---|
| **Logistic Regression** | `class_weight='balanced'` | ⚡ Fast | Moderate | Baseline / interpretability |
| **Random Forest** | `class_weight='balanced'` | 🐢 Slow | Good | Robust, stable predictions |
| **XGBoost** | `scale_pos_weight` | ⚡ Fast | **Best** | Production fraud detection |

> **Key insight:** A model that classifies every transaction as Legit achieves **99.83% accuracy** but catches **zero frauds**.  
> Always use **PR-AUC** and **Recall** as primary metrics for fraud detection.

---
*Credit Card Fraud Detection · Kaggle MLG-ULB dataset · 284,807 transactions · 30 features*